# 00 · Datos, entorno y TWAP determinista

**Taller 3 de RL (MIAX) · Ejecución de una orden CARE de compra.** Este notebook prepara todo lo que usan los ejercicios:
carga los datos de Repsol, presenta el entorno `BestExecutionEnv` con las 13 funciones del `execution_lab.ipynb` completadas,
comprueba sus invariantes y mide los dos puntos de referencia: el **TWAP determinista** (el comportamiento que el ej. 2 debe
replicar por recompensas) y un **agente aleatorio**.

El código vive en `common/` y los notebooks lo importan; cada fichero indica en su cabecera qué deriva del repositorio del
taller (LGPL-2.1) y qué es propio. Las decisiones se registran en [`docs/DECISIONES.md`](../docs/DECISIONES.md).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("."))   # ejecutar desde notebooks/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import data as dt, rewards as rw, exploration as ex, protocol as pr, plots as pl
from common.env import BestExecutionEnv
from common.agents import DDQNAgent, TWAPAgent
pd.set_option("display.precision", 3)
%matplotlib inline

## 1. Datos

`rep_data.pkl` (carpeta `new_version/`, compatible con pandas 2): `{"train", "val", "test"}` → un DataFrame por día con
snapshots del primer nivel del libro cada 5 s entre las 13:00 y las 17:25. Train son los 60 primeros días hábiles de 2018.
Se usa **train** para entrenar y para las figuras, como pide el enunciado del ej. 2.

In [ ]:
ASSET = "REP"                      # D-0.2
DATA = dt.load(ASSET)
train = DATA["train"]
GRID = pr.make_grid(train, n_days=3, seed=0)   # rejilla fija de evaluación (D-0.3)
print(dt.summary(DATA)); print(len(GRID), "episodios de evaluación")

In [ ]:
day = list(train)[0]
df = train[day]
display(df.head())
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
((df.ask1 + df.bid1) / 2).plot(ax=ax[0], title=f"mid {day}")
df.cumvol.diff().clip(lower=0).rolling(60).sum().plot(ax=ax[1], title="volumen negociado (ventana de 5 min)")
plt.tight_layout()

## 2. El entorno

| Nº | Función | Qué hace | Cambio propio |
|---|---|---|---|
| 1 | `_generate_episode_params` | Duración N ~ U{400..600} bins y orden n = int(U[0,075; 0,125] · N) | Forzables |
| 2 | `_generate_episode` | Día y hora de inicio al azar con `look_back` = 60 bins previos | Sorteo directo de una posición válida; forzables |
| 3 | `observation_builder` | Estado [T, V]: tiempo y volumen restantes en tanto por uno | Opción `TVd` (añade V − T) |
| 4 | `_compute_episode_market_feat` | VWAP del episodio con el mid de cada bin y el volumen del bin siguiente | Arrays precalculados |
| 5 | `_compute_market_vwap` | VWAP de mercado acumulado hasta el step | Sumas acumuladas O(1) |
| 6 | `_compute_reward` | Recompensa del paso | Delegada en `rewards.py` |
| 7 | `_agg_action` | Compra 1 título al `ask1` del step | — |
| 8 | `_compute_algo_vwap` | VWAP del algoritmo | `nan` si aún no ha comprado |
| 9 | `_do_nothing` | No compra (precio y volumen 0) | — |
| 10 | `_compute_stop_conditions` | Tiempo agotado / orden completada | — |
| 11 | `_compute_done` | Fin si se cumple cualquiera | — |
| 12 | `_compute_done_reward` | Recompensa final (0 en el original) | Delegada en `rewards.py` |
| 13 | `action_sample` | Acción aleatoria | p = ρ = n/N y generador del agente |

Orden de interacción: `reset()` → `agent.act(s)` → `env.step(a)` (VWAP de mercado → acción → recompensa → `state_pos += 1`
→ `done` → recompensa final) → `agent.experience(...)` → `agent.learn()` cada `learn_after` pasos → `update_target()`.

In [ ]:
import inspect
print(inspect.getsource(BestExecutionEnv.observation_builder))
print(inspect.getsource(BestExecutionEnv.step))

## 3. Invariantes

16 pruebas sobre datos sintéticos con el mismo esquema: rangos de N y n, VWAP calculado a mano, fin por orden completa o por tiempo, el TWAP determinista con error ≤ 1 título, recompensas, checkpoints y ε.

In [ ]:
import unittest
unittest.main(module="common.test_env", argv=["x"], exit=False, verbosity=1)

## 4. Referencias: TWAP determinista y agente aleatorio

El TWAP determinista compra cuando va igual o por detrás del plan (V ≥ T). Es el benchmark de los ejercicios y la definición
operativa de «comportamiento similar» del ej. 2: **error máximo de seguimiento ≤ 1 título** en todos los episodios de la rejilla.

In [ ]:
class RandomAgent:
    epsilon, gamma = 1.0, 1.0
    def __init__(self, env=None, seed=0): self.env, self.rng = env, np.random.default_rng(seed)
    def act(self, s): return self.env.action_sample(rng=self.rng)
    def set_trainable(self, train=False): pass

twap_ev = pr.evaluate(TWAPAgent(), train, GRID)
rand_ev = pr.evaluate(RandomAgent(), train, GRID)
pr.save_df(twap_ev, "00_twap_grid"); pr.save_df(rand_ev, "00_random_grid")
pd.DataFrame({"TWAP determinista": pr.summarize(twap_ev), "aleatorio (p = ρ)": pr.summarize(rand_ev)})

In [ ]:
env = BestExecutionEnv(train, seed=0)
cfg = GRID[4]
pr.run_episode(env, TWAPAgent(env), cfg)
pl.plot_results(env, title=f"TWAP determinista · N={cfg['n_bins']}, n={cfg['vol_care']}", save="00_twap_episodio");

In [ ]:
envs = []
for cfg in GRID[::3]:
    e = BestExecutionEnv(train, seed=0); pr.run_episode(e, RandomAgent(e), cfg); envs.append(e)
pl.plot_trajectories(envs, title="Agente aleatorio (p = ρ): lejos de la diagonal", save="00_random_trayectorias");

## 5. Conclusión

- El entorno cumple las invariantes y el TWAP determinista sigue el plan con error ≤ 1 título en toda la rejilla.
- El agente aleatorio con p = ρ completa la orden en media, pero se separa del plan decenas de títulos: es el punto de
  partida (E0) de la métrica de mejora del ej. 3.